In [19]:
import truststore
truststore.inject_into_ssl()

In [20]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from langchain_community.document_loaders import WebBaseLoader
import bs4

loader = WebBaseLoader(
    web_path = ('https://lilianweng.github.io/posts/2023-06-23-agent/',),
    bs_kwargs=dict(
        parse_only = bs4.SoupStrainer(
            class_ = ('post-content', 'post-title', 'post-header')
        )
    )
)

docs = loader.load()

BM25

In [ ]:
import math
from collections import Counter
from rank_bm25 import BM25Okapi

class BM25Custom:
    def __init__(self, docs: list[str], k=1.5, b=0.75):
        self.k = k
        self.b = b
        self.docs = [self.tokenize(doc) for doc in docs]
        self.N = len(docs)
        self.df = self._build_df()
        self.avgdl = sum(len(d) for d in self.docs) / self.N


    def tokenize(self, doc: str):
        return doc.lower().split()
    

    def _build_df(self):
        df = {}

        for doc in self.docs:
            for text in set(doc):
                df[text] = df.get(text, 0) + 1

        return df
    
    def idf(self, text):
        n = self.df.get(text, 0)
        return math.log(((self.N - n + 0.5)/(n + 0.5))+1)
    
    def score(self, query, doc_idx):
        doc = self.docs[doc_idx]
        dl = len(doc)
        tf_map = Counter(doc)
        total = 0

        for term in self.tokenize(query):
            tf = tf_map.get(term, 0)
            if tf == 0:
                continue

            idf = self.idf(term)
            num = tf * (self.k + 1)
            den = tf + self.k * (1 - self.b + self.b * dl / self.avgdl)
            total += idf * (num / den)
        return total
    
    def get_scores(self, query):
        return [(i, self.score(query, i)) for i in range(self.N)]
    

docs = [
    "Our refund policy allows returns within 30 days",
    "We ship orders within 2 business days",
    "To initiate a refund contact support with order ID",
    "We accept credit cards UPI and net banking",
]

bm25_custom = BM25Custom(docs)
bm25_default = BM25Okapi(docs)

results_custom = bm25_custom.get_scores("refund policy")
results_default = bm25_default.get_scores("refund policy")

Dense Retrieval (Semantic Search)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

docs = [
    "Our refund policy allows returns within 30 days",
    "Car maintenance and vehicle repair services",
    "Automobile engine tune-up and oil change",
    "We ship orders within 2 business days",
    "Privacy policy and data protection",
]

doc_emb = model.encode(docs, normalize_embeddings=True)                 # (5, d)

query = "automobile repair"
query_emb = model.encode([query], normalize_embeddings=True)            # (1, d)

scores = np.dot(doc_emb, query_emb.T).flatten()                         # (1,d) * (d,5) = (1,5)

/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Hybrid Search + RRF

In [ ]:
import weaviate
from weaviate.classes.query import MetadataQuery

client = weaviate.connect_to_local()
collection = client.collections.get("Documents")

results = collection.query.hybrid(
    query="automobile repair",
    alpha=0.5,
    limit=5,
    return_metadata=MetadataQuery(score=True)
)

for obj in results.objects:
    print(obj.metadata.score, obj.properties["content"])

HyDE (Hypothetical Document Embeddings)

In [22]:
# Refer query_transformation for this

Re-ranking (Cross-Encoder)

In [23]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

bi_encoder    = SentenceTransformer('all-MiniLM-L6-v2')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs = [
    "Self-attention computes weighted sum of values using Q-K dot products",
    "Attention allows each token to attend to all other tokens",
    "LSTM uses gates to control information flow across timesteps",
    "Gradient descent minimizes loss by following negative gradient",
    "Transformers replaced RNNs due to parallelism and attention",
    "Refund policy allows returns within 30 days",
]

doc_embs = bi_encoder.encode(docs, normalize_embeddings=True)

query = "What is self-attention in transformers?"
q_emb = bi_encoder.encode([query], normalize_embeddings=True)

bi_scores  = np.dot(doc_embs, q_emb.T).flatten()
top20_idx  = np.argsort(bi_scores)[::-1][:20]
candidates = [docs[i] for i in top20_idx]

pairs = [[query, doc] for doc in candidates]

cross_scores = cross_encoder.predict(pairs)

reranked_idx = np.argsort(cross_scores)[::-1]
final_top3   = [candidates[i] for i in reranked_idx[:3]]

for i, doc in enumerate(final_top3):
    print(f"#{i+1}: {doc}")

/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:1727: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(


#1: Self-attention computes weighted sum of values using Q-K dot products
#2: Transformers replaced RNNs due to parallelism and attention
#3: Attention allows each token to attend to all other tokens
